# Chapter 11 - Control Elements

## Goals

Accelerator models often contain parameters that should change together. Classic Bmad represents these relationships with control elements such as `overlay` and `group`. SciBmad instead provides general parameter relationships and explicit controllers.

This chapter preserves the examples and numerical goals of the original tutorial while expressing them using SciBmad.

In [24]:
using SciBmad
using Printf

## 11.1 Control Elements

A control relationship makes one or more accelerator parameters depend on a smaller set of control variables. SciBmad provides two useful tools:

- `DefExpr` stores a deferred formula and evaluates it when the parameter is requested.
- `Controller` evaluates functions and writes ordinary values when a controller variable changes.

These tools are not tracking elements. Particles do not pass through a `DefExpr` or `Controller`.

## 11.2 Constructing the Example Lattice

We use the same lattice as the original tutorial: a one-meter quadrupole `Q` followed by a one-meter sector bend `B`. The reference particle is a 1 GeV/c muon.

In [25]:
function build_control_lattice()
    q = Quadrupole(name="Q", L=1.0, Kn1=0.0)
    b = SBend(name="B", L=1.0, g_ref=0.0, Kn1=0.0)

    lat = Beamline(
        [q, b];
        species_ref=Species("muon"),
        pc_ref=1e9,
    )

    return (; q, b, lat)
end

base = build_control_lattice();

## 11.3 Displaying the Lattice Structure

SciBmad does not append controllers to a separate lord section. The beamline contains only the elements through which particles are tracked. Control relationships live in parameter values or separate Julia objects.

In [26]:
function show_control_lattice(lat)
    println("Tracked elements:")
    @printf("%5s  %-8s  %-12s  %8s  %8s\n", "index", "name", "kind", "s", "L")
    for ele in lat.line
        @printf("%5d  %-8s  %-12s  %8.3f  %8.3f\n",
            ele.beamline_index, ele.name, ele.kind, ele.s, ele.L)
    end
end

show_control_lattice(base.lat)

Tracked elements:
index  name      kind                 s         L
    1  Q         Quadrupole       0.000     1.000
    2  B         SBend            1.000     1.000


## 11.4 Function-Defined Control with `DefExpr`

This section reproduces the original function-defined control. The control variables `a`, `b`, and `hh` define three magnet parameters:

$$
Q.Kn1 = a+b^2+0.7hh, \qquad Q.x\_offset=0.1hh, \qquad B.g\_ref=0.1a+\tan(b).
$$

For `a=0.02`, `b=0.01`, and `hh=0.01`, the expected values are `Q.Kn1=0.0271`, `Q.x_offset=0.001`, and `B.g_ref=0.012`.

In [33]:
defexpr_example = build_control_lattice()
q_def = defexpr_example.q
b_def = defexpr_example.b

a = 0.02
bvar = 0.01
hh = 0.01

q_def.Kn1 = DefExpr(() -> a + bvar^2 + 0.7*hh)
q_def.x_offset = DefExpr(() -> 0.1*hh)
b_def.g_ref = DefExpr(() -> 0.1*a + tan(bvar));

In [32]:
@printf("Q.Kn1      = %.6f\n", q_def.Kn1)
@printf("Q.x_offset = %.6f m\n", q_def.x_offset)
@printf("B.g_ref    = %.6f 1/m\n", b_def.g_ref)

expected_q_Kn1 = a + bvar^2 + 0.7*hh
expected_q_x_offset = 0.1*hh
expected_b_g_ref = 0.1*a + tan(bvar)

@assert isapprox(q_def.Kn1, expected_q_Kn1)
@assert isapprox(q_def.x_offset, expected_q_x_offset)
@assert isapprox(b_def.g_ref, expected_b_g_ref)

Q.Kn1      = 0.037000
Q.x_offset = 0.001000 m
B.g_ref    = 0.102335 1/m


Changing a captured variable does not immediately write a new number into the magnet. The next time the parameter is requested, the deferred expression is evaluated again.

In [15]:
a = 0.03

@printf("After setting a = 0.03:\n")
@printf("Q.Kn1  = %.6f\n", q_def.Kn1)
@printf("B.g_ref = %.6f 1/m\n", b_def.g_ref)

After setting a = 0.03:
Q.Kn1  = 0.037000
B.g_ref = 0.003000 1/m


## 11.5 Relative Control with `Controller`

The original tutorial next considers the relative-control function

$$
f(z)=0.4\sqrt{z}.
$$

When `z` changes, only the difference `f(z_new)-f(z_old)` is added to `B.Kn1`. SciBmad's generic `Controller` applies the increment, while a small wrapper explicitly stores `old_z`.

In [18]:
controller_example = build_control_lattice()
b_ctrl = controller_example.b

relative_knob = Controller(
    (b_ctrl, :Kn1) => (ele; dKn1) -> ele.Kn1 + dKn1;
    vars=(; dKn1=0.0,),
)

old_z = Ref(0.0)

function set_relative_z!(controller, z, old_z)
    dKn1 = 0.4 * (sqrt(z) - sqrt(old_z[]))
    controller.dKn1 = dKn1
    old_z[] = z
    return dKn1
end;

In [19]:
# z = 0 -> 0.01 adds 0.04.
increment_1 = set_relative_z!(relative_knob, 0.01, old_z)
@printf("First increment:  %.6f; B.Kn1 = %.6f\n", increment_1, b_ctrl.Kn1)

# The bend strength may still be changed directly.
b_ctrl.Kn1 = 0.02

# z = 0.01 -> 0.04 adds another 0.04.
increment_2 = set_relative_z!(relative_knob, 0.04, old_z)
@printf("Second increment: %.6f; B.Kn1 = %.6f\n", increment_2, b_ctrl.Kn1)

@assert isapprox(increment_1, 0.04)
@assert isapprox(increment_2, 0.04)
@assert isapprox(b_ctrl.Kn1, 0.06)

First increment:  0.040000; B.Kn1 = 0.040000
Second increment: 0.040000; B.Kn1 = 0.060000


This reproduces the original relative-control result: after the direct assignment `B.Kn1 = 0.02`, changing `z` from `0.01` to `0.04` adds `0.04`, giving `B.Kn1 = 0.06`.

## 11.6 Example: Chromaticity Control

Consider the situation where you want to control the chromaticity (the change in tune with particle momentum) of a ring by varying sextupole strengths. Assume that a response calculation has already determined how much each sextupole's `Kn2` must change to produce one unit of horizontal chromaticity change. The original response relationship between the sextupole strength and chromaticity may look like

$$
dKn2_{SEX\_08W}=-6.415\times10^{-3}\,d\xi_x.
$$

In a real ring, all coefficients would come from a chromaticity-response calculation.

In [20]:
@elements begin
    sex_08w = Sextupole(name="SEX_08W", L=0.20, Kn2=1.20)
    sex_10w = Sextupole(name="SEX_10W", L=0.20, Kn2=1.10)
    sex_12w = Sextupole(name="SEX_12W", L=0.20, Kn2=-1.00)
    sex_14w = Sextupole(name="SEX_14W", L=0.20, Kn2=-1.15)
end

chromaticity_section = Beamline(
    [sex_08w, sex_10w, sex_12w, sex_14w];
    species_ref=Species("muon"),
    pc_ref=1e9,
)

x_chromaticity_knob = Controller(
    (sex_08w, :Kn2) => (ele; dxi_step) -> ele.Kn2 - 6.415e-3 * dxi_step,
    (sex_10w, :Kn2) => (ele; dxi_step) -> ele.Kn2 - 5.870e-3 * dxi_step,
    (sex_12w, :Kn2) => (ele; dxi_step) -> ele.Kn2 + 4.920e-3 * dxi_step,
    (sex_14w, :Kn2) => (ele; dxi_step) -> ele.Kn2 + 5.360e-3 * dxi_step;
    vars=(; dxi_step=0.0,),
)

old_dxi_x = Ref(0.0)

function set_horizontal_chromaticity_change!(controller, dxi_x, old_dxi_x)
    controller.dxi_step = dxi_x - old_dxi_x[]
    old_dxi_x[] = dxi_x
end;

In [21]:
initial_Kn2 = [sex_08w.Kn2, sex_10w.Kn2, sex_12w.Kn2, sex_14w.Kn2]

# Request a one-unit horizontal chromaticity change.
set_horizontal_chromaticity_change!(x_chromaticity_knob, 1.0, old_dxi_x)

final_Kn2 = [sex_08w.Kn2, sex_10w.Kn2, sex_12w.Kn2, sex_14w.Kn2]
changes = final_Kn2 .- initial_Kn2

for (ele, dKn2) in zip((sex_08w, sex_10w, sex_12w, sex_14w), changes)
    @printf("%-8s  dKn2 = % .6e  new Kn2 = % .6f\n", ele.name, dKn2, ele.Kn2)
end

@assert isapprox(changes[1], -6.415e-3)

sex_08w   dKn2 = -6.415000e-03  new Kn2 =  1.193585
sex_10w   dKn2 = -5.870000e-03  new Kn2 =  1.094130
sex_12w   dKn2 =  4.920000e-03  new Kn2 = -0.995080
sex_14w   dKn2 =  5.360000e-03  new Kn2 = -1.144640


One change of the chromaticity target updates the entire sextupole family. The wrapper applies only the change from the previous target, reproducing the relative behavior of the original group controller. The response coefficients translate that chromaticity change into magnet-strength changes.

The chapter examples now follow the original sequence:

- Section 11.4: parameters defined by functions of control variables;
- Section 11.5: relative control using `0.4sqrt(z)`;
- Section 11.6: a coordinated sextupole knob for changing chromaticity.

## 11.7 Exercises

1. **Control from knot points.** A control function does not need to be written as a single analytic expression. Construct a knot-point controller that uses cubic-spline interpolation between specified points. Over the interval `-0.04 <= hh <= 0.04`, use knot points to reproduce the Section 11.4 relationships

   $$
   Q.Kn1=0.7hh, \qquad Q.x\_offset=0.1hh.
   $$

   Evaluate the interpolated control at values between the knots and compare it with the analytic relationships. Define how the controller behaves outside the knot interval.

2. **Move a bend while preserving lattice length.** Construct a simple lattice containing an upstream drift `D`, a bend `B`, and a quadrupole `Q`. Add a controller that moves the upstream edge of `B` longitudinally while keeping the downstream edge of `B`, and therefore the total lattice length, fixed. This requires changing `D.L` and `B.L` in tandem. Verify the upstream and downstream positions of `B` and the total lattice length before and after the move.